# Where should I look for stellar oscillations?

This notebook predicts global properties of solar-like oscillations: the characteristic frequency $\nu_{\max}$, the approximate frequency range around it, the large separation $\Delta\nu$, and a photometric radial-mode amplitude. Solar-like oscillations can be observed in both radial velocity and photometry, but AsteroScale currently predicts only photometric amplitudes.

In the following we'll just use AsteroScale to make point estimates. You can of course supply measurement uncertainties to obtain properly marginalized distributions.

Here we assume that mass, radius, effective temperature, and metallicity are known. Because they are supplied without uncertainties, the calculation returns central point predictions.

In [6]:
import asteroscale as ast

star = {"M": 1.1, "R": 1.3, "Teff": 6000, "FeH": 0.1}

prediction = ast.solve(
    star,
    want=["numax", "dnu", "FWHM_env", "A_env",],
    bandpass="TESS",
)
prediction

{'numax': np.float64(1982.2491458473373),
 'dnu': np.float64(94.85782037346993),
 'FWHM_env': np.float64(598.0358941076695),
 'A_env': np.float64(2.9642737456372936)}

The broad oscillation-power envelope is approximated by a Gaussian centred on `numax`. In a power spectrum from, for example, TESS photometry, a useful first search interval is `numax ± FWHM_env/2`. This is not a hard boundary, and the individual mode peaks form a structured pattern inside it. `A_env` is the maximum radial-mode RMS amplitude, not the total light-curve RMS or the envelope's integrated power.

In [2]:
lower = prediction["numax"] - prediction["FWHM_env"] / 2
upper = prediction["numax"] + prediction["FWHM_env"] / 2
print(f"Suggested search interval: {lower:.0f}–{upper:.0f} microhertz")

Suggested search interval: 1683–2281 microhertz


## TESS and Kepler amplitudes

Oscillation amplitudes decrease in redder photometric bandpasses. The predicted Kepler amplitude is therefore slightly larger than the TESS amplitude because Kepler's response is bluer.

In [4]:
for bandpass in ("TESS", "Kepler"):
    amplitude = ast.solve(star, want=["A_env"], bandpass=bandpass)["A_env"]
    print(f"{bandpass:6s}: {amplitude:.2f} ppm")

TESS  : 2.96 ppm
Kepler: 3.53 ppm


## Hot stars have lower amplitudes 

For stars hotter than the Sun, the empirical amplitude decreases near the red edge of the classical instability strip. This transition roughly separates cooler solar-like oscillators from hotter $\gamma$ Doradus and $\delta$ Scuti pulsators. 

**Caution:** AsteroScale does not model $\gamma$ Doradus or $\delta$ Scuti pulsations; those modes are excited by different physics. A very small solar-like amplitude at high temperature is not a prediction of the other pulsation spectrum.

In [9]:
star = {"M": 1.1, "R": 1.3, "Teff": 7200, "FeH": 0.1}

prediction = ast.solve(
    star,
    want=["numax", "dnu", "FWHM_env", "A_env"],
    bandpass="TESS",
)
prediction

{'numax': np.float64(1809.537619626558),
 'dnu': np.float64(90.63755013598447),
 'FWHM_env': np.float64(901.5058834654263),
 'A_env': np.float64(1.9495766102104717)}

A predicted signal is not necessarily detectable or important for the variance in a light curve. Compare its frequency range with the observations' Nyquist frequency and frequency resolution, and compare an instrument-appropriate signal model with the local instrumental and stellar background.